In [2]:
import pandas as pd

placement = pd.read_csv("../data/processed/placement_cleaned.csv")
notes = pd.read_csv("../data/raw/company_notes_placement_agent.csv")
notes.columns

Index(['Company Name', 'Interview Process / Rounds', 'Role Type',
       'Approximate CGPA Cutoff', 'Sample Technical / Round Questions',
       'Sample HR / Behavioral Questions', 'Evidence / Notes'],
      dtype='str')

In [3]:
notes.columns = [c.strip() for c in notes.columns]

print(placement.shape, notes.shape)
placement.head()

(803, 6) (13, 7)


,year,raw_company_name,clean_company,n_selected,ctc_lpa,ctc_needs_review
0,2016-17,"TCS Limited, Hyderabad",TCS,192.0,NaN,False
1,2016-17,Huyndai Mobis,Huyndai Mobis,1.0,NaN,False
2,2016-17,Nalsoft,Nalsoft,3.0,NaN,False
3,2016-17,Tech Mahindra,Tech Mahindra,42.0,NaN,False
4,2016-17,NTT Data,NTT Data,29.0,NaN,False


# Turn each row into a text "document" to embed

Two different chunk types, since they answer different question types:

In [4]:
def stats_chunk(row):
    ctc = f"{row['ctc_lpa']} LPA" if pd.notna(row['ctc_lpa']) else "CTC not recorded"
    n = int(row['n_selected']) if pd.notna(row['n_selected']) else "unknown number of"
    return f"In {row['year']}, {row['clean_company']} selected {n} students at {ctc}."

def notes_chunk(row):
    return (
        f"{row['Company Name']} interview process: {row['Interview Process / Rounds']}. "
        f"Role type: {row['Role Type']}. "
        f"Technical questions: {row['Sample Technical / Round Questions']}. "
        f"HR questions: {row['Sample HR / Behavioral Questions']}."
    )

stats_docs = [stats_chunk(r) for _, r in placement.iterrows()]
notes_docs = [notes_chunk(r) for _, r in notes.iterrows()]
print(stats_docs[0])
print(notes_docs[0])

In 2016-17, TCS selected 192 students at CTC not recorded.
TCS interview process: Online Assessment (TCS NQT / role-specific test) → Technical → Managerial/HR. Role type: Software development / analyst; role varies. Technical questions: Technical: Explain your project; OOP concepts; DBMS/SQL; write/trace a basic program. HR questions: Managerial/HR: Tell me about yourself; Are you willing to relocate?; Why TCS?.


# Embed and store in ChromaDB

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../data/processed/chroma_db")
collection = client.get_or_create_collection("placement_data")

r:\aiml course iit patna\projects\Campus_Placement_Prep_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3543.47it/s]


In [6]:
all_docs = stats_docs + notes_docs
all_ids = [f"stat_{i}" for i in range(len(stats_docs))] + [f"note_{i}" for i in range(len(notes_docs))]
all_metadata = (
    [{"type":"stat", "company": r["clean_company"]} for _, r in placement.iterrows()]
    + [{"type":"notes", "company": r["Company Name"]} for _, r in notes.iterrows()]
)

embeddings = model.encode(all_docs).tolist()
collection.add(
    documents=all_docs, embeddings = embeddings, ids = all_ids, metadatas = all_metadata
)
print(collection.count())

816


# Test retrieval before building anything else on top

In [7]:
def search(query, k=5):
    q_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"[{meta['type']}] {doc}")
    print()

search("What's the interview process for TCS?")
search("How many students did Amazon select?")
search("What technical questions does Cognizant ask?")

[notes] TCS interview process: Online Assessment (TCS NQT / role-specific test) → Technical → Managerial/HR. Role type: Software development / analyst; role varies. Technical questions: Technical: Explain your project; OOP concepts; DBMS/SQL; write/trace a basic program. HR questions: Managerial/HR: Tell me about yourself; Are you willing to relocate?; Why TCS?.
[notes] QSpiders interview process: Aptitude/technical assessment → interview; exact stages vary by drive. Role type: Software testing / development training-placement. Technical questions: Technical: programming basics; OOP; SQL/DBMS; testing fundamentals. HR questions: HR: introduction; relocation; career goals.
[notes] EPAM interview process: Online Assessment / coding → Technical interview(s) → HR. Role type: Software development. Technical questions: Technical: DSA; OOP; programming language fundamentals; project/system discussion. HR questions: HR: motivation; teamwork; career goals; communication.
[notes] Wipro interview

In [1]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

notes = pd.read_csv("../data/raw/company_notes_placement_agent.csv")
notes.columns = [c.strip() for c in notes.columns]

FIELD_MAP = {
    "interview_process": "Interview Process / Rounds",
    "role_type": "Role Type",
    "cgpa_cutoff": "Approximate CGPA Cutoff",
    "technical_questions": "Sample Technical / Round Questions",
    "hr_questions": "Sample HR / Behavioral Questions",
}

def is_blank(v):
    return pd.isna(v) or str(v).strip() == "" or str(v).strip().lower() in ("nan", "n/a", "none")

def make_field_chunks(row):
    company = row["Company Name"].strip()
    chunks = []
    for field_key, col in FIELD_MAP.items():
        val = row.get(col)
        if is_blank(val):
            continue
        val = str(val).strip()
        prefix = {
            "interview_process": "interview process / rounds",
            "role_type": "typically hires for role type",
            "cgpa_cutoff": "approximate CGPA cutoff",
            "technical_questions": "sample technical / round interview questions",
            "hr_questions": "sample HR / behavioral interview questions",
        }[field_key]
        chunks.append({
            "text": f"{company} {prefix}: {val}",
            "metadata": {"company": company, "field": field_key},
        })
    return chunks

all_chunks = []
for _, row in notes.iterrows():
    all_chunks.extend(make_field_chunks(row))

print(f"{len(all_chunks)} chunks from {len(notes)} companies")

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../data/processed/chroma_db")

# drop any stale collection from earlier runs (old mixed stats+notes schema)
try:
    client.delete_collection("placement_data")
except Exception:
    pass

collection = client.get_or_create_collection("company_notes")

docs = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"{m['company']}_{m['field']}" for m in metadatas]

embeddings = model.encode(docs).tolist()
collection.add(documents=docs, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(collection.count())

def search(query, k=5, field=None):
    q_emb = model.encode([query]).tolist()
    where = {"field": field} if field else None
    results = collection.query(query_embeddings=q_emb, n_results=k, where=where)
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"[{meta['field']}] {doc}")
    print()

search("What's the interview process for TCS?")
search("What technical questions does Cognizant ask?", field="technical_questions")

r:\aiml course iit patna\projects\Campus_Placement_Prep_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


65 chunks from 13 companies


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2480.82it/s]


65
[technical_questions] TCS sample technical / round interview questions: Technical: Explain your project; OOP concepts; DBMS/SQL; write/trace a basic program
[interview_process] TCS interview process / rounds: Online Assessment (TCS NQT / role-specific test) → Technical → Managerial/HR
[interview_process] QSpiders interview process / rounds: Aptitude/technical assessment → interview; exact stages vary by drive
[role_type] TCS typically hires for role type: Software development / analyst; role varies
[interview_process] Accenture interview process / rounds: Cognitive & Technical Assessment → Coding → Communication → Technical + HR

[technical_questions] Cognizant sample technical / round interview questions: Technical: Which programming language are you comfortable with?; Swap two numbers without temporary variable; aptitude problem solving
[technical_questions] Divi's Laboratories sample technical / round interview questions: Technical questions depend strongly on role/branch; prepar